# A 연습 노트북 — `basemul` CPA 직접 구현

원본 분석 노트북은 그대로 두고, 이 파일에서 **한 단계씩 코드를 직접 작성**해 봐. 원본 `mlkem-cpa-basemul.ipynb`는 참고/정답 확인용이고, 여기는 일부러 TODO가 남아 있는 실습지야.

## 오늘의 목표
- 저장된 암호문과 그 파형을 행별로 짝짓는 이유를 이해한다.
- 후보 비밀계수마다 leakage model을 만들고 Pearson CPA 점수와 rank를 계산한다.
- 구현 산술 검증과 키 복구 평가, blind 여부를 구분한다.

셀을 위에서부터 진행해. TODO 셀은 네가 채운 다음 실행하면 돼. 파일명은 고정해뒀어.


## 0. 캡처 데이터 읽기

`traces[t]`와 `ciphertexts[t]`는 같은 decapsulation에서 나온 한 쌍이야. CPA에서는 이 대응이 깨지면 후보 예측값과 파형이 다른 실험끼리 비교돼서 상관이 무의미해져.

이 파일은 **알려진 키가 포함된 기존 1,000-trace 캡처**야. 우선 shape만 확인하고 `dk`나 실제 계수 값은 출력하지 마. 실제 키는 이 실습에서 rank 채점에만 쓰도록 할 거야.


In [ ]:
import numpy as np

with np.load("../traces/mlkem768-clean-O3-basemul-20261030-042431.npz") as data:
    traces = data["traces"].astype(np.float32)
    ciphertexts = data["ct"].astype(np.uint8)
    dk = data["dk"].tobytes()

print("traces:", traces.shape, "ciphertexts:", ciphertexts.shape)


### 체크포인트 0

예상 shape은 `traces=(1000, 125000)`, `ciphertexts=(1000, 1088)`야. 첫 축의 1000이 같은 순서의 캡처 개수고, 둘째 축은 파형 샘플 수 또는 ciphertext 바이트 수야.

**질문:** 왜 파형을 하나로 평균내면 안 될까? → 각 트레이스의 ciphertext가 다르므로 각 행마다 다른 후보 중간값이 대응돼.


## 1. 실험 설정

`a1 = ŝ[VEC][2P+1]`를 공격해. `b1`은 각 공개 암호문에서 계산되는 값이고, `a1`은 캡처 동안 고정된 비밀계수야. 처음엔 지정된 좌표로 결과를 재현하고, 그다음 `P`를 바꿔 실험해.


In [ ]:
Q, N, K = 3329, 256, 3
VEC, P = 0, 7
WINDOW = (1900, 1940)


## 2. MCU 산술을 Python으로 재현하기 — 직접 작성

정답 셀을 바로 복사하지 말고, 원본 노트북의 산술 셀을 참고해 함수들을 직접 입력해 봐. 이 부분은 `int16_t` wrap과 Montgomery/Barrett reduction을 재현해.

- `to_i16`: signed 16-bit 범위로 wrap
- `montgomery_reduce`: PQClean의 Montgomery reduction
- `fqmul`: 두 값 곱한 뒤 Montgomery reduction
- `barrett_reduce`: NTT 후 사용하는 Barrett reduction

이 함수들이 틀리면 이후 모델의 Hamming weight도 틀릴 수 있어. 단순히 값이 `mod q`에서 같은지만 확인하지 말고, 16-bit 표현을 유지해.


In [ ]:
def to_i16(value):
    # TODO: signed int16 wrap을 구현해.
    raise NotImplementedError


def montgomery_reduce(value):
    # TODO: 원본 노트북의 PQClean 연산 순서를 옮겨 적어.
    raise NotImplementedError


def fqmul(left, right):
    # TODO
    raise NotImplementedError


def barrett_reduce(value):
    # TODO
    raise NotImplementedError


### 체크포인트 1

위 함수를 채운 뒤 아래 기본 경계값 테스트를 통과시켜.


In [ ]:
assert to_i16(32768) == -32768
assert to_i16(-32769) == 32767
print("int16 wrap OK")


## 3. 암호문에서 CPA 입력 `b1` 만들기 — 직접 작성

원본 노트북의 `ntt`, `polyvec_decompress_vec`, `bhat_from_ct`를 참고해 필요한 함수들을 작성해. 흐름은 `ct`의 `u` 압축 해제 → NTT → 선택 좌표 읽기야. `dk`에서 `ŝ`를 파싱하는 `poly_frombytes`/`shat_from_dk`도 정답 rank 확인에 필요해.

구현 전체가 부담되면 원본 함수 하나씩 따라 적고, 각 함수가 어떤 포맷을 입력·출력하는지 주석으로 설명해 봐.


In [ ]:
# TODO: 원본 노트북을 참고해 다음 함수를 작성해.
# ntt(coefficients)
# poly_frombytes(packed_poly)
# polyvec_decompress_vec(packed_u)
# bhat_from_ct(ciphertext)
# shat_from_dk(decapsulation_key)


### 체크포인트 2 — 타깃 값과 알려진 입력

다음 셀을 채워서 `true_a1`과 캡처별 `b1`을 준비해. `b1`은 파형 행과 같은 순서로 만들어야 해.


In [ ]:
# TODO:
# true_a1 = ...
# b1 = np.array([... for ct in ciphertexts], dtype=np.int64)

assert b1.shape == (len(traces),)
print("b1 shape:", b1.shape)


## 4. 후보별 누출 예측값 만들기 — 직접 작성

각 후보 `g∈{0,…,3328}`와 트레이스 `t`에 대해 `MontgomeryReduce(g × b1[t])`의 **16-bit 비트 패턴** Hamming weight를 계산해 `hw_model[g,t]`를 만들어. 결과 shape은 `(3329,1000)`이야.

**주의:** 앞에서 만든 scalar `montgomery_reduce`는 내부에서 `int(value)`를 사용해. `(3329,1000)` NumPy 배열에 그대로 호출하면 에러가 나므로, 원본 단일계수 셀의 `montgomery_reduce_vec`처럼 배열용 함수를 별도로 작성해. `np.uint16`으로 비트 패턴을 해석한 다음 lookup table로 HW를 얻어.


In [ ]:
# TODO: 원본 단일계수 셀을 참고해 montgomery_reduce_vec(values)를 작성해.
# TODO: key_guesses, HW16_TABLE, hw_model을 작성해.
# 기대 shape: (Q, len(traces))
assert hw_model.shape == (Q, len(traces))


### 체크포인트 3 — 상관과 rank

후보 모델의 트레이스 축 평균을 빼고, 파형에서도 각 샘플의 트레이스 축 평균을 빼. 그다음 `corr[g, sample]`을 계산해.

각 후보 점수는 선택한 `WINDOW` 안의 `max(abs(corr))`야. 정답 계수의 rank는 그 점수보다 높은 후보 수로 구해. 채점용 정답은 후보 모델 생성에 넣지 않고, 마지막에 rank만 확인해.


In [ ]:
# TODO:
# 1) trace_window = traces[:, WINDOW[0]:WINDOW[1]]
# 2) centered 모델/파형과 Pearson 상관 행렬 계산
# 3) 후보별 max |corr|, peak sample, true_rank 계산

assert corr.shape == (Q, WINDOW[1] - WINDOW[0])
print("best guess:", best_guess, "rank:", true_rank, "peak:", peak_sample)


## 5. 결과 해석과 확장 과제

기존 캡처의 지정 조건에서는 단일계수 known-key 평가가 rank 0에 도달해야 해. 그래도 이건 전체 키 blind 복구가 아니야. 창을 미리 고정했고, 실제 계수로 사후 rank를 채점하며, 주변 계수의 영향과 다른 `basemul` 연산에서 생기는 ghost peak도 고려해야 해.

**추가 연습**
1. `WINDOW`를 넓혔을 때 최고 피크가 어디로 이동하는지 비교해.
2. `P`를 바꿔 후보 점수와 peak가 어떻게 변하는지 기록해.
3. 앞의 20/50/100/200개 트레이스만 사용해 rank 곡선을 만들어.
4. 타깃 키를 모르는 공격을 주장하려면 어떤 선택(POI, 부호, 창)이 키 정답을 참조했는지 목록으로 써.

완성 코드는 원본 `mlkem-cpa-basemul.ipynb`와 비교해. 원본의 full-key sweep은 정답 기반 부호·피크 보정이 포함된 oracle 기준선이야.
